# Markdown-per-Page → Chunked

Nimmt die Markdown-per-Page JSONs aus `report_merging.ipynb` und füllt das `content.chunks`-Feld mit token-basierten Chunks (für Vector-DB-Ingestion).

## Einstellungen

| Parameter | Wert |
|---|---|
| Splitter | `RecursiveCharacterTextSplitter.from_tiktoken_encoder` |
| Encoding | `cl100k_base` |
| `chunk_size` | 300 Tokens |
| `chunk_overlap` | 20 Tokens |

## Verarbeitung

- Chunking erfolgt **pro Seite** — Chunks überspannen keine Seitengrenzen.
- Leere Seiten werden übersprungen.
- `chunk_id` ist fortlaufend pro Dokument (0..N).
- Jeder Chunk referenziert seine Quellseite über `page`.

## Output-Schema

```json
{
  "metainfo": { /* identisch zu Stufe 2 */ },
  "content": {
    "chunks": [
      { "chunk_id": 0, "page": 1, "text": "..." },
      { "chunk_id": 1, "page": 1, "text": "..." }
    ],
    "pages": [ /* identisch zu Stufe 2 */ ]
  }
}
```

In [1]:
import json
from pathlib import Path
from typing import List, Dict, Optional
from tqdm import tqdm
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [2]:
class Chunker:
    """Markdown-per-page JSON -> chunked JSON for vector DB."""

    def __init__(self, source_dir: str, output_dir: Optional[str] = None,
                 chunk_size: int = 300, chunk_overlap: int = 20,
                 encoding_name: str = "cl100k_base"):
        self.source_dir = Path(source_dir)
        if output_dir:
            self.output_dir = Path(output_dir)
        else:
            base = self.source_dir.name
            if base.endswith("_merged"):
                base = base[:-len("_merged")]
            self.output_dir = self.source_dir.parent / (base + "_chunked")
        self.output_dir.mkdir(parents=True, exist_ok=True)

        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
            encoding_name=encoding_name,
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        )

        print(f"Source: {self.source_dir}")
        print(f"Output: {self.output_dir}")
        print(f"Chunk size: {chunk_size} tokens, overlap: {chunk_overlap}")

    # -- Page chunking ----------------------------------------------------

    def _chunk_page(self, page: Dict, next_chunk_id: int) -> List[Dict]:
        text = page.get("text", "")
        if not text.strip():
            return []

        page_num = page["page"]
        pieces = self.splitter.split_text(text)
        return [
            {
                "chunk_id": next_chunk_id + i,
                "page": page_num,
                "text": piece,
            }
            for i, piece in enumerate(pieces)
        ]

    # -- Document processing ----------------------------------------------

    def _process_document(self, data: Dict) -> Dict:
        pages = data["content"]["pages"]
        all_chunks: List[Dict] = []

        for page in pages:
            page_chunks = self._chunk_page(page, len(all_chunks))
            all_chunks.extend(page_chunks)

        return {
            "metainfo": data["metainfo"],
            "content": {
                "chunks": all_chunks,
                "pages": pages,
            },
        }

    # -- Batch processing -------------------------------------------------

    def process_all(self) -> Dict:
        json_files = sorted(self.source_dir.glob("*.json"))
        print(f"{len(json_files)} JSON files\n")

        stats = {"success": 0, "error": 0, "chunks_total": 0}

        for jf in tqdm(json_files, desc="Chunking"):
            try:
                with open(jf, "r", encoding="utf-8") as f:
                    data = json.load(f)

                result = self._process_document(data)

                out_file = self.output_dir / jf.name
                with open(out_file, "w", encoding="utf-8") as f:
                    json.dump(result, f, indent=2, ensure_ascii=False)

                num_chunks = len(result["content"]["chunks"])
                num_pages = len(result["content"]["pages"])
                stats["success"] += 1
                stats["chunks_total"] += num_chunks
                tqdm.write(f"  {jf.stem}: {num_pages} pages -> {num_chunks} chunks")

            except Exception as e:
                tqdm.write(f"  {jf.stem}: Error \u2014 {e}")
                stats["error"] += 1

        print(f"\nOK: {stats['success']} | Errors: {stats['error']} | Chunks: {stats['chunks_total']}")
        return stats

In [3]:
chunker = Chunker(
    source_dir="C:/Users/phili/PycharmProjects/doc/data/src/docling_output/20260409_195022_merged",
    chunk_size=300,
    chunk_overlap=20,
)
chunker.process_all()

Source: C:\Users\phili\PycharmProjects\doc\data\src\docling_output\20260409_195022_merged
Output: C:\Users\phili\PycharmProjects\doc\data\src\docling_output\20260409_195022_chunked
Chunk size: 300 tokens, overlap: 20
48 JSON files



Chunking:   2%|▏         | 1/48 [00:00<00:11,  4.20it/s]

  CE_2012: 159 pages -> 671 chunks


Chunking:   4%|▍         | 2/48 [00:00<00:11,  4.04it/s]

  CE_2013: 172 pages -> 689 chunks


...

Chunking:  98%|█████████▊| 47/48 [00:09<00:00,  5.71it/s]

  DVN_2007: 116 pages -> 536 chunks


Chunking: 100%|██████████| 48/48 [00:09<00:00,  4.86it/s]

  DVN_2010: 154 pages -> 649 chunks

OK: 48 | Errors: 0 | Chunks: 26345


{'success': 48, 'error': 0, 'chunks_total': 26345}